# 01. 하루치 경향 살펴보기

**목적**: 변수들이 하루 동안 어떻게 움직이는지 눈으로 확인하고, 불량이 난 지점에서 무슨 일이 있었는지 본다.

**쓰는 데이터**: `labeled_data.csv` (1차 가공본)
- 시각, LH/RH, 불량유형이 살아 있어서 진단에 쓴다
- 2차 가공본은 표준화되어 있어 물리 단위가 안 보이므로 탐색에는 안 쓴다

**사용법**: 아래 `날짜`와 `제품` 만 바꿔가며 여러 날을 보면 된다.

In [ ]:
import sys
sys.path.append("..")

from src.load import load_labeled, day_slice, GROUPS
from src.plotting import setup, save, plot_group
import matplotlib.pyplot as plt

setup()

## 1. 데이터 읽기

In [ ]:
df = load_labeled("CN7")   # "RG3" 로 바꾸면 다른 제품

print("행 수:", len(df))
print("불량:", df.y.sum(), "건")
print("기간:", df.ts.min(), "~", df.ts.max())
print()
print("불량 유형별:")
print(df[df.y == 1].Reason.value_counts())

## 2. 어느 날을 볼지 고르기

불량이 있는 날부터 보는 게 좋다. 아래 목록에서 하나 골라 다음 셀의 `날짜` 에 넣는다.

In [ ]:
불량일 = df[df.y == 1].groupby("date").agg(
    불량수=("y", "sum"),
    유형=("Reason", lambda s: ", ".join(s.unique()))
).sort_values("불량수", ascending=False)

불량일

## 3. 하루치 자르기

In [ ]:
날짜 = "2020-10-16"        # ← 여기만 바꾸면 된다

day = day_slice(df, 날짜)
print(f"{날짜}: {len(day)}행, 쇼트 {day.shot.nunique()}개, 불량 {day.y.sum()}건")
day[["ts", "side", "PassOrFail", "Reason", "shot_idx"]].head(10)

In [ ]:
import pandas as pd

df  = load_labeled("CN7")          # "CN7" 로 바꿔 한 번 더
bad = df[df["y"] == 1]

# 1) 유형별 총 개수
print("── 유형별 총계 ──")
print(bad["Reason"].value_counts().to_string())
print(f"합계 {len(bad)}건\n")

# 2) 유형 × LH/RH
print("── 유형 × LH/RH ──")
print(pd.crosstab(bad["Reason"], bad["side"],
                  margins=True, margins_name="계").to_string(), "\n")

t = (bad.groupby(["date", "Reason", "side"]).size()
        .unstack(["Reason", "side"], fill_value=0))
t["계"] = t.sum(axis=1)
print(t.to_string())

## 4. 금형온도 3번과 4번

두 채널이 하루 동안 어떻게 움직이는지, 둘이 같이 움직이는지 본다.

**볼 것**
- 기동 직후 올라가다가 평평해지는가 (열평형)
- 3번과 4번의 차이가 일정한가, 아니면 벌어졌다 좁아졌다 하는가
- 불량 지점(점선)에서 뭔가 달랐는가

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_group(day, GROUPS["금형온도"], f"{날짜} 금형온도", ax=ax)
ax.set_ylabel("℃")
save(fig, f"{날짜}_금형온도")
plt.show()

# 두 채널의 차이만 따로
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(day.ts, day.Mold_Temperature_4 - day.Mold_Temperature_3, lw=0.9, color="purple")
ax.axhline(0, color="gray", lw=0.5)
ax.set_title(f"{날짜} 금형온도 4번 − 3번")
ax.set_ylabel("℃")
ax.grid(alpha=0.3)
plt.show()

## 5. 배럴온도 6구간

**볼 것**
- 1번(노즐쪽)이 가장 높고 6번(호퍼쪽)이 가장 낮은 층 구조가 유지되는가
- 어느 한 구간만 튀는 시점이 있는가

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_group(day, GROUPS["배럴온도"], f"{날짜} 배럴온도", ax=ax)
ax.set_ylabel("℃")
save(fig, f"{날짜}_배럴온도")
plt.show()

## 6. 시간 변수

**볼 것**
- Injection_Time 과 Filling_Time 의 차이 = 가압+보압 구간. 이게 줄어든 시점이 있는가
- Plasticizing_Time(계량)이 길어지면 Cycle_Time 도 길어지는가 (계량이 병목인지)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

plot_group(day, ["Injection_Time", "Filling_Time"], f"{날짜} 사출/충전 시간", ax=axes[0])
axes[0].set_ylabel("초")

axes[1].plot(day.ts, day.Injection_Time - day.Filling_Time, lw=0.9, color="green")
axes[1].set_title("사출 − 충전 (= 가압+보압 추정)")
axes[1].set_ylabel("초")
axes[1].grid(alpha=0.3)

plt.tight_layout()
save(fig, f"{날짜}_시간변수")
plt.show()

## 7. 압력

**볼 것**
- Max_Injection_Pressure 가 어떤 값에서 딱 잘린 것처럼 멈추는가 (= 상한 설정)
- Max_Switch_Over_Pressure(절환 압력)가 낮은 시점에 미성형이 났는가

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_group(day, GROUPS["압력"], f"{날짜} 압력", ax=ax)
ax.set_ylabel("MPa")
save(fig, f"{날짜}_압력")
plt.show()

## 8. 쇼트 순번으로 보기

시각 대신 "그날 몇 번째 쇼트인가"를 가로축에 놓으면 기동 직후 구간이 잘 보인다.
초기허용불량이 26~30번째에 몰린다는 가설을 여기서 확인한다.

In [ ]:
lh = day[day.side == "LH"].sort_values("shot_idx")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(lh.shot_idx, lh.Mold_Temperature_3, lw=0.9, label="금형온도 3")
ax.plot(lh.shot_idx, lh.Mold_Temperature_4, lw=0.9, label="금형온도 4")
for _, r in lh[lh.y == 1].iterrows():
    ax.axvline(r.shot_idx, color="red", ls="--", alpha=0.7)
ax.set_xlabel("그날 몇 번째 쇼트")
ax.set_ylabel("℃")
ax.set_title(f"{날짜} 쇼트 순번 기준 (빨간선 = 불량)")
ax.legend()
ax.grid(alpha=0.3)
save(fig, f"{날짜}_쇼트순번")
plt.show()

## 9. 같은 쇼트의 LH와 RH 비교

가설: 같은 쇼트면 공정값이 완전히 같다. 정말 그런지 확인한다.
만약 미세하게 다르다면 캐비티별 신호가 데이터에 남아 있다는 뜻이라 중요하다.

In [ ]:
from src.load import PROC_COLS

쌍 = day.groupby("shot")[PROC_COLS].nunique().max()
print("같은 쇼트 안에서 값이 2종류 이상인 변수:")
print(쌍[쌍 > 1] if (쌍 > 1).any() else "  없음 — 모든 변수가 LH/RH 동일")

## 10. 발견한 것 적기

뭔가 보이면 `docs/findings.md` 에 한 줄 추가한다.

```
- [09-24] [이름] 10/16 금형온도 3번이 오전 내내 상승하다 11시경 안정 → reports/figures/2020-10-16_금형온도.png
```

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.load import load_labeled

d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)
d["온도차"] = d["Mold_Temperature_3"] - d["Mold_Temperature_4"]

print("온도차 > 0 인 비율")
print(d.groupby("prod")["온도차"].apply(lambda s: (s > 0).mean()).round(3).to_string())
print()
print(d.groupby("prod")["온도차"].describe()[["mean","std","min","max"]].round(2).to_string())

In [ ]:
lab = load_labeled("CN7")
day = lab[lab["date"].astype(str) == "2020-10-27"].copy()   # 초기허용불량 10건이 난 날
day["경과분"] = (day["ts"] - day["ts"].min()).dt.total_seconds() / 60
day["온도차"] = day["Mold_Temperature_3"] - day["Mold_Temperature_4"]

import matplotlib.pyplot as plt
from src.plotting import setup, save
setup()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(day["경과분"], day["온도차"], lw=.8, c="#2c7fb8")
ax.axhline(-1.42, ls="--", c="gray", label="전체 평균 -1.42")
for _, r in day[day["y"] == 1].iterrows():
    ax.axvline(r["경과분"], c="crimson", alpha=.6, lw=1)
ax.set_xlim(0, 180)
ax.set_xlabel("런 시작 후 분"); ax.set_ylabel("Mold_3 − Mold_4")
ax.set_title("10-27 기동 — 금형 상하 온도차와 불량 시점(빨간 선)")
ax.legend(); ax.grid(alpha=.3)
save(fig, "열평형_10-27"); plt.show()

# 경과 시간대별 온도차
import numpy as np
b = pd.cut(day["경과분"], [0,10,20,30,45,60,90,120,10000],
           labels=["~10","10~20","20~30","30~45","45~60","60~90","90~120","120+"])
print(day.groupby(b)["온도차"].agg(["mean","std","count"]).round(2).to_string())

In [ ]:
d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)
d["사출행정"] = d["Plasticizing_Position"] - d["Cushion_Position"]

print(d.groupby("prod")["사출행정"].describe()[["mean","std","min","max"]].round(2).to_string())

# 정상 대비 편차 (제품별 표준화)
m = d.groupby("prod")["사출행정"].transform("mean")
s = d.groupby("prod")["사출행정"].transform("std")
d["행정편차"] = (d["사출행정"] - m) / s

print("\n── 불량 유형별 행정편차 ──")
print(d.groupby(["prod","Reason"])["행정편차"]
       .agg(["count","mean","std"]).round(2).to_string())
print("\n양품 평균:", d[d.y==0].groupby("prod")["행정편차"].mean().round(3).to_string())

In [ ]:
d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)
d["온도차"] = d["Mold_Temperature_3"] - d["Mold_Temperature_4"]
d["plan"] = pd.to_datetime(d["PART_FACT_PLAN_DATE"].astype(str)
                            .str.split().str[0]).dt.date

g = d.groupby(["plan","prod"]).agg(
        온도차=("온도차","mean"), 금형3=("Mold_Temperature_3","mean"),
        금형4=("Mold_Temperature_4","mean"),
        행=("y","size"), 불량=("y","sum"))
g["불량률%"] = (g["불량"]/g["행"]*100).round(2)
print(g.round(2).to_string())

In [ ]:
print(d.groupby("prod")[["Plasticizing_Position","Cushion_Position"]]
       .describe().T.round(2).to_string())
print("\n고유값 개수")
print(d[["Plasticizing_Position","Cushion_Position"]].nunique().to_string())
print("\nCushion 상위 값들")
print(d["Cushion_Position"].value_counts().head(12).to_string())

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.load import load_labeled, PROC_COLS

d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)

rows = []
for c in PROC_COLS:
    for p in ["CN7", "RG3"]:
        s = d.loc[d["prod"] == p, c]
        mu = s.mean()
        rows.append(dict(변수=c, 제품=p, 고유값=s.nunique(),
                         평균=mu, 표준편차=s.std(),
                         변동계수=(s.std()/abs(mu) if mu else np.nan)))

a = pd.DataFrame(rows)
a["판정"] = np.where(a["고유값"] <= 1, "상수(사용불가)",
            np.where(a["변동계수"] < 0.001, "사실상 고정",
            np.where(a["고유값"] <= 3, "이산(소수값)", "사용가능")))

print(a.pivot(index="변수", columns="제품", values=["고유값","변동계수","판정"])
       .round(4).to_string())
print("\n── 판정 집계 ──")
print(a.groupby(["제품","판정"]).size().to_string())
print("\n── 제품별 사용가능 변수 ──")
for p in ["CN7","RG3"]:
    ok = a[(a["제품"]==p) & (a["판정"]=="사용가능")]["변수"].tolist()
    print(f"{p} ({len(ok)}개): {ok}")

In [ ]:
import numpy as np, pandas as pd
d["plan"] = pd.to_datetime(d["PART_FACT_PLAN_DATE"].astype(str)
                            .str.split().str[0]).dt.date

rows = []
for p in ["CN7", "RG3"]:
    s = d[d["prod"] == p]
    for c in PROC_COLS:
        tot = s[c].std()
        if not tot or np.isnan(tot):
            rows.append(dict(제품=p, 변수=c, 런간비중=np.nan,
                             런내불량편차=np.nan)); continue
        g = s.groupby("plan")[c]
        within = (s[c] - g.transform("mean")).std()          # 런 내부 변동
        icc = 1 - (within**2) / (tot**2)                     # 런 간이 차지하는 비중

        sd = g.transform("std").replace(0, np.nan)
        z = (s[c] - g.transform("mean")) / sd                # 런 내부 표준화
        rows.append(dict(제품=p, 변수=c, 런간비중=icc,
                         런내불량편차=z[s["y"] == 1].mean()))

t = pd.DataFrame(rows)
t["판정"] = np.where(t["런간비중"].isna(), "상수",
            np.where(t["런간비중"] > 0.9, "런 단위(위험)",
            np.where(t["런간비중"] > 0.6, "혼합", "쇼트 단위(양호)")))

print(t.pivot(index="변수", columns="제품",
              values=["런간비중","런내불량편차","판정"]).round(3).to_string())
print("\n── 집계 ──")
print(t.groupby(["제품","판정"]).size().to_string())

In [ ]:
rows = []
for p in ["CN7", "RG3"]:
    s = d[d["prod"] == p]
    for c in PROC_COLS:
        g = s.groupby("plan")[c]
        sd = g.transform("std").replace(0, np.nan)
        z = (s[c] - g.transform("mean")) / sd
        for rsn in ["가스", "미성형", "초기허용불량"]:
            m = s["Reason"] == rsn
            if m.sum() == 0: continue
            rows.append(dict(제품=p, 변수=c, 유형=rsn,
                             n=int(m.sum()), 편차=z[m].mean()))

u = pd.DataFrame(rows)
piv = u.pivot_table(index="변수", columns=["제품","유형"], values="편차").round(2)
print(piv.to_string())

print("\n── 유형별 최대 |편차| 변수 ──")
for (p, r), s2 in u.groupby(["제품","유형"]):
    top = s2.reindex(s2["편차"].abs().sort_values(ascending=False).index).head(3)
    print(f"\n{p} {r} (n={top['n'].iloc[0]})")
    print(top[["변수","편차"]].to_string(index=False))

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

use = [c for c in PROC_COLS if d[c].nunique() > 3]

for p in ["CN7", "RG3"]:
    s = d[d["prod"] == p].copy()
    g = s.groupby("plan")
    Z = pd.DataFrame({c: (s[c] - g[c].transform("mean")) /
                         g[c].transform("std").replace(0, np.nan)
                      for c in use}).fillna(0)

    s["점수"] = Z.abs().max(axis=1)          # 가장 많이 벗어난 변수의 정도
    ap = average_precision_score(s["y"], s["점수"])
    print(f"\n{p}  AP={ap:.3f}  (무작위 기준 {s['y'].mean():.4f})")

    for rsn in ["초기허용불량", "미성형", "가스"]:
        m = s["Reason"] == rsn
        if m.sum() == 0: continue
        # 상위 1% 안에 들어오는 비율
        thr = s["점수"].quantile(0.99)
        print(f"  {rsn:8s} n={m.sum():3d}  상위1% 검출률 {(s.loc[m,'점수']>=thr).mean():.0%}")

In [ ]:
def causal_score(s, cols):
    """직전 쇼트들까지만 써서 표준화 — 실시간 적용 가능한 형태"""
    s = s.sort_values("ts")
    Z = {}
    for c in cols:
        g = s.groupby("plan")[c]
        mu = g.transform(lambda x: x.expanding().mean().shift())
        sd = g.transform(lambda x: x.expanding().std().shift())
        Z[c] = ((s[c] - mu) / sd.replace(0, np.nan))
    return pd.DataFrame(Z).abs().max(axis=1)

for p in ["CN7", "RG3"]:
    s = d[d["prod"] == p].copy()
    s["점수_causal"] = causal_score(s, use).fillna(0)
    ap = average_precision_score(s["y"], s["점수_causal"])
    print(f"{p} 인과형 AP={ap:.3f} (기저 {s['y'].mean():.4f}, "
          f"{ap/s['y'].mean():.0f}배)")


In [ ]:
import matplotlib.pyplot as plt
from src.plotting import setup, save
setup()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, p in zip(axes, ["CN7", "RG3"]):
    s = d[d["prod"] == p].copy()
    g = s.groupby("plan")
    Z = pd.DataFrame({c: (s[c]-g[c].transform("mean"))
                         / g[c].transform("std").replace(0, np.nan)
                      for c in use}).fillna(0)
    s["점수"] = Z.abs().max(axis=1)

    s = s.sort_values("점수", ascending=False)
    x = np.arange(1, len(s)+1) / len(s) * 100
    y = s["y"].cumsum() / s["y"].sum() * 100

    ax.plot(x, y, lw=2, c="#2c7fb8", label="런내 이탈도 기준")
    ax.plot([0,100],[0,100], ls="--", c="gray", label="무작위 검사")
    for q in [1, 5, 10]:
        i = int(len(s)*q/100)
        ax.plot(q, y.iloc[i], "o", c="crimson")
        ax.annotate(f"{q}% → {y.iloc[i]:.0f}%", (q, y.iloc[i]),
                    xytext=(8, -10), textcoords="offset points", fontsize=9)
    ax.set_xlim(0, 30); ax.set_ylim(0, 100)
    ax.set_xlabel("검사 비율 (%)"); ax.set_ylabel("불량 검출률 (%)")
    ax.set_title(f"{p}"); ax.grid(alpha=.3); ax.legend(loc="lower right")

fig.suptitle("검사 우선순위 곡선 — 상위 몇 %를 검사하면 불량의 몇 %가 잡히는가")
plt.tight_layout(); save(fig, "검사우선순위곡선"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
s = d[d["prod"] == "CN7"].copy()
g = s.groupby("plan")
Z = pd.DataFrame({c: (s[c]-g[c].transform("mean"))
                     / g[c].transform("std").replace(0, np.nan)
                  for c in use}).fillna(0)
s["점수"] = Z.abs().max(axis=1)
s = s.sort_values("점수", ascending=False).reset_index(drop=True)
x = (s.index + 1) / len(s) * 100

for rsn, col in [("초기허용불량","#f39c12"), ("미성형","#2c7fb8"), ("가스","#c0392b")]:
    hit = (s["Reason"] == rsn)
    ax.plot(x, hit.cumsum() / hit.sum() * 100, lw=2, c=col,
            label=f"{rsn} (n={hit.sum()})")
ax.plot([0,100],[0,100], ls="--", c="gray", lw=1, label="무작위")
ax.set_xlim(0,30); ax.set_ylim(0,100)
ax.set_xlabel("검사 비율 (%)"); ax.set_ylabel("해당 유형 검출률 (%)")
ax.set_title("CN7 — 불량 유형별 검사 우선순위 곡선")
ax.grid(alpha=.3); ax.legend(loc="lower right")
plt.tight_layout(); save(fig, "검사우선순위_유형별"); plt.show()

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.load import load_labeled, PROC_COLS

if "d" not in dir() or "run" not in d.columns:
    d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)
    d["plan"] = pd.to_datetime(d["PART_FACT_PLAN_DATE"].astype(str)
                                .str.split().str[0]).dt.date
    d = d.sort_values("ts").reset_index(drop=True)
    _gap = d.groupby(["plan","prod"])["ts"].diff().dt.total_seconds() / 60
    d["run"] = ((_gap > 60) | _gap.isna()).cumsum()
    print(f"d 재생성: {len(d):,}행, 런 {d['run'].nunique()}개")

use = [c for c in PROC_COLS if d[c].nunique() > 3]

top = ["Max_Injection_Speed","Max_Switch_Over_Pressure","Filling_Time",
       "Injection_Time","Cushion_Position","Max_Injection_Pressure"]

s = d[d["prod"] == "CN7"].copy()
g = s.groupby("run")
for c in top:
    z = (s[c] - g[c].transform("mean")) / g[c].transform("std").replace(0, np.nan)
    print(f"\n── {c} ──")
    print(f"양품     n={int((s.y==0).sum()):4d}  평균 {z[s.y==0].mean():+.2f}")
    for rsn in ["초기허용불량","미성형","가스"]:
        m = s["Reason"] == rsn
        if m.sum() == 0: continue
        v = z[m].dropna().sort_values()
        print(f"{rsn:8s} n={len(v):2d}  평균 {v.mean():+6.2f}  중앙 {v.median():+6.2f}"
              f"  범위 [{v.min():+.2f}, {v.max():+.2f}]")
        print(f"          {[round(x,1) for x in v]}")

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.load import load_labeled, PROC_COLS, RAW
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# d 없으면 생성
if "d" not in dir() or "run" not in d.columns:
    d = pd.concat([load_labeled("CN7"), load_labeled("RG3")], ignore_index=True)
    d["plan"] = pd.to_datetime(d["PART_FACT_PLAN_DATE"].astype(str)
                                .str.split().str[0]).dt.date
    d = d.sort_values("ts").reset_index(drop=True)
    _g = d.groupby(["plan","prod"])["ts"].diff().dt.total_seconds() / 60
    d["run"] = ((_g > 60) | _g.isna()).cumsum()
use = [c for c in PROC_COLS if d[c].nunique() > 3]

# 비라벨 읽기 (같은 설비만)
u = pd.read_csv(RAW / "unlabeled_data.csv",
                usecols=["PART_NAME", "EQUIP_NAME"] + PROC_COLS)
u = u[u["EQUIP_NAME"] == "650톤-우진2호기"].copy()
u["prod"] = u["PART_NAME"].str.extract(r"^(\w+)")[0]
print(f"비라벨 {len(u):,}행  (CN7 {(u['prod']=='CN7').sum():,} / "
      f"RG3 {(u['prod']=='RG3').sum():,})\n")

# 도메인 분류기: 라벨된 행인지 아닌지를 맞혀본다
for p in ["CN7", "RG3"]:
    a = d.loc[d["prod"] == p, use].assign(라벨됨=1)
    bb = u.loc[u["prod"] == p, use].dropna()
    b = bb.sample(min(20000, len(bb)), random_state=42).assign(라벨됨=0)
    X = pd.concat([a, b], ignore_index=True)
    auc = cross_val_score(
        RandomForestClassifier(200, n_jobs=-1, random_state=42),
        X[use], X["라벨됨"], cv=3, scoring="roc_auc").mean()
    print(f"{p}  라벨 {len(a):,} vs 비라벨 {len(b):,}  →  AUC = {auc:.3f}")

In [ ]:
u2 = pd.read_csv(RAW / "unlabeled_data.csv",
                 usecols=["PART_NAME","EQUIP_NAME","PART_FACT_PLAN_DATE"] + PROC_COLS)
u2 = u2[u2["EQUIP_NAME"] == "650톤-우진2호기"].copy()
u2["prod"] = u2["PART_NAME"].str.extract(r"^(\w+)")[0]
u2["plan"] = pd.to_datetime(u2["PART_FACT_PLAN_DATE"].astype(str)
                             .str.split().str[0]).dt.date

lab_days = set(d["plan"])
same = u2[u2["plan"].isin(lab_days)]
print(f"라벨된 12일의 비라벨 행: {len(same):,}\n")

for p in ["CN7", "RG3"]:
    a  = d.loc[d["prod"] == p, use].assign(라벨됨=1)
    bb = same.loc[same["prod"] == p, use].dropna()
    if len(bb) < 200:
        print(f"{p}: 같은 날 비라벨이 {len(bb)}행뿐 — 비교 불가"); continue
    b = bb.sample(min(20000, len(bb)), random_state=42).assign(라벨됨=0)
    X = pd.concat([a, b], ignore_index=True)
    auc = cross_val_score(RandomForestClassifier(200, n_jobs=-1, random_state=42),
                          X[use], X["라벨됨"], cv=3, scoring="roc_auc").mean()
    print(f"{p}  라벨 {len(a):,} vs 같은날 비라벨 {len(b):,}  →  AUC = {auc:.3f}")

In [ ]:
print(u["PART_NAME"].value_counts().to_string())
print()
other = u[~u["prod"].isin(["CN7","RG3"])]
print(f"기타 {len(other):,}행")
print(other["prod"].value_counts().to_string())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for p in ["CN7", "RG3"]:
    a  = d.loc[d["prod"] == p, use].assign(라벨됨=1)
    bb = same.loc[same["prod"] == p, use].dropna()
    if len(bb) < 200: continue
    b = bb.sample(min(20000, len(bb)), random_state=42).assign(라벨됨=0)
    X = pd.concat([a, b], ignore_index=True)

    m = RandomForestClassifier(200, n_jobs=-1, random_state=42).fit(X[use], X["라벨됨"])
    imp = pd.Series(m.feature_importances_, index=use).sort_values(ascending=False)
    print(f"\n── {p}: 라벨/비라벨을 가르는 변수 상위 6 ──")
    print(imp.head(6).round(3).to_string())

In [ ]:
from sklearn.metrics import average_precision_score

u2["key"] = list(zip(u2["plan"], u2["prod"]))
d["key"]  = list(zip(d["plan"],  d["prod"]))

for p in ["CN7", "RG3"]:
    lab = d[d["prod"] == p]
    unl = u2[(u2["prod"] == p) & (u2["plan"].isin(set(lab["plan"])))]

    for 이름, base in [("라벨만", lab[["key"] + use]),
                      ("비라벨 포함", pd.concat([lab[["key"] + use], unl[["key"] + use]]))]:
        st = base.groupby("key")[use].agg(["mean", "std"])
        Z = pd.DataFrame(index=lab.index)
        for c in use:
            mu = lab["key"].map(st[(c, "mean")])
            sd = lab["key"].map(st[(c, "std")]).replace(0, np.nan)
            Z[c] = (lab[c] - mu) / sd
        sc = Z.abs().max(axis=1).fillna(0)
        ap = average_precision_score(lab["y"], sc)
        print(f"{p}  {이름:10s} AP={ap:.3f}  ({ap/lab['y'].mean():.1f}배)")
    print()

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score

uu = pd.read_csv(RAW / "unlabeled_data.csv",
                 usecols=["TimeStamp","PART_NAME","EQUIP_NAME",
                          "PART_FACT_PLAN_DATE"] + PROC_COLS)
uu = uu[uu["EQUIP_NAME"] == "650톤-우진2호기"].copy()
uu["prod"] = uu["PART_NAME"].str.extract(r"^(\w+)")[0]
uu["ts"]   = pd.to_datetime(uu["TimeStamp"])
uu["plan"] = pd.to_datetime(uu["PART_FACT_PLAN_DATE"].astype(str)
                             .str.split().str[0]).dt.date
uu["y"] = np.nan

for p in ["CN7", "RG3"]:
    lab = d[d["prod"] == p].copy()
    unl = uu[(uu["prod"] == p) & (uu["plan"].isin(set(lab["plan"])))].copy()

    allrows = (pd.concat([lab.assign(라벨=1), unl.assign(라벨=0)], ignore_index=True)
                 .sort_values("ts").reset_index(drop=True))

    Z = pd.DataFrame(index=allrows.index)
    g = allrows.groupby("plan")
    for c in use:
        mu = g[c].transform(lambda x: x.expanding(min_periods=30).mean().shift())
        sd = g[c].transform(lambda x: x.expanding(min_periods=30).std().shift())
        Z[c] = (allrows[c] - mu) / sd.replace(0, np.nan)
    allrows["점수"] = Z.abs().max(axis=1)

    ev = allrows[(allrows["라벨"] == 1) & allrows["점수"].notna()]
    ap = average_precision_score(ev["y"], ev["점수"])
    print(f"{p}  인과형+비라벨  AP={ap:.3f}  기저 {ev['y'].mean():.4f}  "
          f"→ {ap/ev['y'].mean():.1f}배   (채점 {len(ev):,}/{int((allrows['라벨']==1).sum()):,}행)")

In [ ]:
def variant(p, 비라벨, 인과, minp=30):
    lab = d[d["prod"] == p].copy()
    if 비라벨:
        unl = uu[(uu["prod"] == p) & (uu["plan"].isin(set(lab["plan"])))].copy()
        base = pd.concat([lab.assign(라벨=1), unl.assign(라벨=0)], ignore_index=True)
    else:
        base = lab.assign(라벨=1)
    base = base.sort_values("ts").reset_index(drop=True)

    Z = pd.DataFrame(index=base.index)
    g = base.groupby("plan")
    for c in use:
        if 인과:
            mu = g[c].transform(lambda x: x.expanding(min_periods=minp).mean().shift())
            sd = g[c].transform(lambda x: x.expanding(min_periods=minp).std().shift())
        else:
            mu, sd = g[c].transform("mean"), g[c].transform("std")
        Z[c] = (base[c] - mu) / sd.replace(0, np.nan)
    base["점수"] = Z.abs().max(axis=1)
    return base[(base["라벨"] == 1) & base["점수"].notna()]

print(f"{'방식':28s} {'CN7':>18s} {'RG3':>18s}")
for 이름, (비, 인) in {
    "런 전체통계 · 라벨만":   (False, False),
    "런 전체통계 · 비라벨포함": (True,  False),
    "인과형(min30) · 라벨만":  (False, True),
    "인과형(min30) · 비라벨포함":(True,  True)}.items():
    out = []
    for p in ["CN7", "RG3"]:
        ev = variant(p, 비, 인)
        ap = average_precision_score(ev["y"], ev["점수"])
        out.append(f"{ap:.3f} ({ap/ev['y'].mean():4.1f}배)")
    print(f"{이름:28s} {out[0]:>18s} {out[1]:>18s}")

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score

d  = d.loc[:,  ~d.columns.duplicated()].copy()
uu = uu.loc[:, ~uu.columns.duplicated()].copy()

def variant(prod, with_unlab, causal, minp=30):
    lab = d.loc[d["prod"] == prod].copy()
    lab["is_lab"] = 1
    parts = [lab]
    if with_unlab:
        unl = uu.loc[(uu["prod"] == prod) &
                     (uu["plan"].isin(set(lab["plan"])))].copy()
        unl["is_lab"] = 0
        if "y" not in unl.columns:
            unl["y"] = np.nan
        parts.append(unl)

    cols = ["ts", "plan", "y", "is_lab"] + use
    if "Reason" in lab.columns:
        cols.append("Reason")
    base = pd.concat([x.reindex(columns=cols) for x in parts], ignore_index=True)
    base = base.sort_values("ts").reset_index(drop=True)

    g = base.groupby("plan")
    Z = pd.DataFrame(index=base.index)
    for c in use:
        if causal:
            mu = g[c].transform(lambda x: x.expanding(min_periods=minp).mean().shift())
            sd = g[c].transform(lambda x: x.expanding(min_periods=minp).std().shift())
        else:
            mu = g[c].transform("mean")
            sd = g[c].transform("std")
        Z[c] = (base[c] - mu) / sd.replace(0, np.nan)
    base["score"] = Z.abs().max(axis=1)

    return base.loc[(base["is_lab"] == 1) & base["score"].notna()].copy()


VARIANTS = [
    ("런 전체통계 · 라벨만",      False, False),
    ("런 전체통계 · 비라벨포함",   True,  False),
    ("인과형(min30) · 라벨만",    False, True),
    ("인과형(min30) · 비라벨포함", True,  True),
]

print(f"{'방식':26s} {'CN7':>20s} {'RG3':>20s}")
print("-" * 70)
for name, wu, ca in VARIANTS:
    cells = []
    for prod in ["CN7", "RG3"]:
        ev = variant(prod, wu, ca)
        ap = average_precision_score(ev["y"], ev["score"])
        cells.append(f"{ap:.3f} ({ap/ev['y'].mean():4.1f}배, n={len(ev):,})")
    print(f"{name:26s} {cells[0]:>20s} {cells[1]:>20s}")

In [ ]:
ev  = variant("CN7", False, True)                 # 최종 모델
thr = ev["score"].quantile(0.99)

cn7 = d.loc[d["prod"] == "CN7"]
tot = cn7.groupby("Reason").size()

print(f"채점된 불량 {int(ev['y'].sum())}건 / 전체 {int(cn7['y'].sum())}건")
print(f"채점 행 {len(ev):,} / {len(cn7):,}\n")

for rsn in ["초기허용불량", "미성형", "가스"]:
    m   = ev["Reason"] == rsn
    hit = (ev.loc[m, "score"] >= thr).mean() if m.sum() else 0
    print(f"{rsn:8s} 채점 {m.sum():2d}/{int(tot.get(rsn, 0)):2d}건   "
          f"상위1% 검출 {hit:.0%}")

# 채점에서 빠진 불량이 뭔지
missed = cn7[(cn7["y"] == 1) & (~cn7.index.isin(ev.index))]
if len(missed):
    print(f"\n── 채점 안 된 불량 {len(missed)}건 ──")
    print(missed[["ts","side","Reason","shot_idx"]].to_string(index=False))

In [ ]:
def prev_run_variant(prod, minrows=100):
    """직전 런(라벨·비라벨 무관)의 통계로 이번 런을 채점"""
    lab = d.loc[d["prod"] == prod].copy();  lab["is_lab"] = 1
    unl = uu.loc[uu["prod"] == prod].copy(); unl["is_lab"] = 0
    if "y" not in unl.columns: unl["y"] = np.nan
    if "Reason" not in unl.columns: unl["Reason"] = np.nan

    cols = ["ts", "plan", "y", "Reason", "is_lab"] + use
    allr = (pd.concat([lab.reindex(columns=cols), unl.reindex(columns=cols)],
                      ignore_index=True)
              .sort_values("ts").reset_index(drop=True))

    runs = sorted(allr["plan"].unique())
    allr["score"] = np.nan
    ref = None
    for r in runs:
        cur = allr["plan"] == r
        if ref is not None:
            mu, sd = ref
            z = (allr.loc[cur, use] - mu) / sd.replace(0, np.nan)
            allr.loc[cur, "score"] = z.abs().max(axis=1)
        if cur.sum() >= minrows:                      # 다음 런의 기준으로 삼는다
            ref = (allr.loc[cur, use].mean(), allr.loc[cur, use].std())

    return allr.loc[(allr["is_lab"] == 1) & allr["score"].notna()].copy()


for prod in ["CN7", "RG3"]:
    ev = prev_run_variant(prod)
    ap = average_precision_score(ev["y"], ev["score"])
    print(f"\n{prod}  직전런 기준  AP={ap:.3f}  ({ap/ev['y'].mean():.1f}배)  "
          f"채점 {len(ev):,}행 / 불량 {int(ev['y'].sum())}건")
    thr = ev["score"].quantile(0.99)
    for rsn in ["초기허용불량", "미성형", "가스"]:
        m = ev["Reason"] == rsn
        if m.sum() == 0: continue
        print(f"   {rsn:8s} 채점 {m.sum():2d}건  상위1% 검출 "
              f"{(ev.loc[m,'score']>=thr).mean():.0%}")

In [ ]:
import pandas as pd
from src.load import RAW, load_labeled

print("── 1차 가공본 ──")
lab1 = pd.read_csv(RAW / "labeled_data.csv")
print(f"labeled_data.csv            {len(lab1):,}행, {lab1.shape[1]}열")
p = lab1["PART_NAME"].str.extract(r"^(\w+)")[0]
print(p.value_counts().to_string())

print("\n── 2차 가공본 ──")
for f in ["moldset_labeled_cn7", "moldset_labeled_rg3", "moldset_labeled",
          "moldset_unlabeled_cn7", "moldset_unlabeled_rg3", "supervised_label_cn7"]:
    try:
        x = pd.read_csv(RAW / f"{f}.csv", nrows=5)
        n = sum(1 for _ in open(RAW / f"{f}.csv", encoding="utf-8")) - 1
        print(f"{f:26s} {n:>9,}행, {x.shape[1]:2d}열  {list(x.columns)[:5]}")
    except Exception as e:
        print(f"{f:26s} 읽기 실패: {e}")

print("\n── 1차 CN7/RG3 (중복 포함/제거) ──")
for dd in [False, True]:
    for prod in ["CN7", "RG3"]:
        n = len(load_labeled(prod, drop_dup=dd))
        print(f"  drop_dup={dd!s:5s} {prod}: {n:,}행")

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.load import RAW

FILES = {"CN7": "moldset_labeled_cn7.csv", "RG3": "moldset_labeled_rg3.csv"}
VARS = [c for c in pd.read_csv(RAW / FILES["CN7"], nrows=1).columns
        if c not in ("Unnamed: 0", "PassOrFail")]

def load_lab(prod):
    lab = (pd.read_csv(RAW / FILES[prod])
             .rename(columns={"Unnamed: 0": "order", "PassOrFail": "y"})
             .sort_values("order").reset_index(drop=True))
    k = pd.util.hash_pandas_object(lab[VARS], index=False).values
    pid = np.empty(len(lab), dtype=int); p = i = 0
    while i < len(lab):
        pid[i] = p
        if i + 1 < len(lab) and k[i] == k[i + 1]:
            pid[i + 1] = p; i += 2
        else:
            i += 1
        p += 1
    lab["pair"] = pid
    return lab

# 외부데이터: 공식 라벨 파일의 원본
ml = pd.read_csv(RAW / "moldset_labeled.csv")
print("moldset_labeled:", len(ml), "행 | Unnamed:0 고유값", ml["Unnamed: 0"].nunique(),
      "| 범위", ml["Unnamed: 0"].min(), "~", ml["Unnamed: 0"].max())
m = ml.drop_duplicates("Unnamed: 0").set_index("Unnamed: 0")

for prod in ["CN7", "RG3"]:
    lab = load_lab(prod)
    lab["side"] = lab["order"].map(m["PART_NAME"].str.extract(r"(LH|RH)")[0])
    y_ml = lab["order"].map(m["PassOrFail"])
    if y_ml.dtype == object:
        y_ml = (y_ml == "N").astype(float)

    print(f"\n── {prod} ──")
    print(f"좌우 못 붙인 행: {lab['side'].isna().sum()}")
    print(f"라벨 불일치:     {(lab['y'] != y_ml).sum()}")
    two = lab.groupby("pair")["pair"].transform("size") == 2
    s = lab[two].groupby("pair")["side"].nunique()
    print(f"쌍 중 좌우가 갈린 쌍: {(s == 2).sum()} / {len(s)}")
    print(pd.crosstab(lab["side"], lab["y"], margins=True).to_string())

In [ ]:
x = ml[ml["Unnamed: 0"] >= 2393]
print(x["PART_NAME"].value_counts().to_string())
print(x["TimeStamp"].min(), "~", x["TimeStamp"].max())
print(x["PassOrFail"].value_counts().to_string())